# Phase 0 — Hindi voice-cloning bake-off

Runtime → **Change runtime type** → GPU (T4 is enough; L4/A100 is faster).

Run the cells in order. Everything is resumable: re-running a render cell skips
WAVs that already exist, and `phase0/out` survives a runtime restart.

**Read the install notes in section 2 before running anything else** — Colab and
chatterbox disagree about which torch to use, and that is the one thing that
reliably breaks this notebook.

In [ ]:
!nvidia-smi -L
import sys, torch
print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Drive + harness

The harness lives in Drive at `MyDrive/audio_automation/phase0`, next to the
reference clip. It is copied to local Colab disk so renders and `out/` stay fast;
only the report and the zip go back to Drive.

GitHub clone is the fallback when the Drive copy is missing.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess, glob

DRIVE_DIR = "/content/drive/MyDrive/audio_automation"

if not os.path.isdir(f"{DRIVE_DIR}/phase0"):
    found = (glob.glob("/content/drive/MyDrive/*/audio_automation")
             + glob.glob("/content/drive/Shareddrives/*/audio_automation"))
    if found:
        DRIVE_DIR = found[0]
        print("audio_automation is not at MyDrive root; using", DRIVE_DIR)

REF = f"{DRIVE_DIR}/hindi_10s.wav"
LOCAL = "/content/voice_clone"
REPO = "https://github.com/itz-learningtime-oss/voice_clone.git"
BRANCH = "main"

if os.path.isdir(f"{DRIVE_DIR}/phase0"):
    os.makedirs(LOCAL, exist_ok=True)
    subprocess.run(["rsync", "-a", "--exclude", "out", "--exclude", "__pycache__",
                    f"{DRIVE_DIR}/phase0/", f"{LOCAL}/phase0/"], check=True)
    print("harness: Drive")
else:
    if not os.path.isdir(f"{LOCAL}/.git"):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, LOCAL],
                       check=True)
    print("harness: GitHub")

if os.path.isdir(f"{DRIVE_DIR}/docs"):
    os.makedirs(f"{LOCAL}/docs", exist_ok=True)
    subprocess.run(["rsync", "-a", f"{DRIVE_DIR}/docs/", f"{LOCAL}/docs/"], check=True)
    print("docs: Drive")

os.chdir(LOCAL)
print("cwd:", os.getcwd())

if not os.path.exists(REF):
    from google.colab import files
    REF = list(files.upload().keys())[0]
print("reference:", REF)

## 2. Install, repair, restart

`chatterbox-tts` pins `torch==2.6.0` / `torchaudio==2.6.0` / `numpy<2`, but Colab
ships torch 2.11 + torchvision cu130 and numpy 2.x. If you let pip resolve freely
you get `torchvision requires torch==2.11.0, but you have torch 2.6.0` and broken
imports.

So: install, then force the torch trio and a compatible numpy (the pin depends on
the Python version), then **restart the session once** (binary wheels are only
loaded at process start), then run `doctor`.

The `gcsfs` / `starlette` / `google-adk` / `datasets` / `ipython` conflict lines in
pip's output are Colab's own packages and do not affect this run. Only a torch,
torchaudio, torchvision or numpy line matters.

In [ ]:
!pip install -q -r phase0/requirements.txt

In [ ]:
# torch 2.6.0 + matching torchvision/torchaudio, and a numpy that both
# chatterbox and numba accept (the pin depends on the Python version).
import sys, subprocess

numpy_pin = "numpy==1.26.4" if sys.version_info < (3, 13) else "numpy==2.2.6"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
    numpy_pin, "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
    "--extra-index-url", "https://download.pytorch.org/whl/cu124",
], check=True)
print("done — now Runtime -> Restart session, then re-run cell 3 (Drive + harness)")

### After the restart

Re-run **cell 3** (Drive + harness) so `REF`/`LOCAL` exist again, then run the
doctor cell. It must say `OK       stack looks consistent` before you render.

In [ ]:
!python phase0/run.py doctor

## 3. Validate the reference

The clip is 10 s of the target speaker taken from `video.mp3` (48 kHz mono,
8 % silence). Confirm it is the right voice and has no music underneath.

In [ ]:
!python phase0/run.py check --reference "$REF"

## 4. Reference transcript

Chatterbox does not need it; IndicF5 does. The app will do this automatically in
Phase 2, so this also tests the ASR round-trip.

**Read the output and fix any wrong word before running the next cell.** A wrong
transcript silently ruins IndicF5's clone. Also confirm the language is Hindi.

In [ ]:
from faster_whisper import WhisperModel

asr = WhisperModel("large-v3", device="cuda", compute_type="float16")
segments, info = asr.transcribe(REF, language="hi", vad_filter=False, beam_size=5)
draft = " ".join(segment.text.strip() for segment in segments).strip()
print("detected language:", info.language, f"(p={info.language_probability:.2f})")
print(draft)

REF_TEXT = draft  # edit this string by hand if the ASR made a mistake
print()
print("using:", REF_TEXT)

## 5. Render the two Chatterbox variants

`cb_mtl_configB` is the primary target; `cb_mtl_default` isolates how much the
Config B recipe actually buys. Both use the same weights, so this is a clean A/B.

In [ ]:
!python phase0/run.py render --reference "$REF" \
  --candidates cb_mtl_configB,cb_mtl_default

## 6. Second sweep: spoken numbers + T3 v3

First sweep said: Config B is 3x more intelligible than the defaults, but two
engine flaws remain. This sweep isolates them.

- `cb_mtl_configB_spoken` — same weights and settings as the baseline, but
  digits are spelled out in Hindi first (`24,07,000` → `चौबीस लाख सात हज़ार`).
  Only `n02` changes, so the delta is unambiguous.
- `cb_mtl_v3_configB` — pins the T3 **v3** checkpoint. `from_pretrained()`
  defaults to v2, so the first sweep may have measured v2. Downloads ~2 GB
  once; `timings.csv` now records which checkpoint actually loaded.

Both are resumable and skip existing WAVs.

In [ ]:
!python phase0/run.py render --reference "$REF" \
  --candidates cb_mtl_configB_spoken,cb_mtl_v3_configB

## 7. Hindi finetune (the one that should sound native)

`ResembleAI/Chatterbox-Multilingual-hi` is the official Hindi pack: T3 `hi`
weights (292 tensors, 2.1 GB) finetuned from the V3 checkpoint.

Weights are loaded by hand into the base model (`t3.load_state_dict`, strict
first, then a key-level report) because chatterbox 0.1.7 has no
`from_pretrained(t3_model=)` or `from_local()` — that is why the first attempt
was skipped. `s3gen_v3.pt` is loaded alongside the V3 T3; set
`PHASE0_NO_S3GEN_SWAP=1` to keep the base decoder.

`timings.csv` records what actually loaded (`t3=t3_hi (strict); s3gen=v3`), and
`report.md` now prints that under **Loaded checkpoints**.

In [ ]:
!python phase0/run.py render --reference "$REF" --candidates cb_hi_configB

## 8. Score

Whisper is loaded once for all files. Transcripts, speaker embeddings and UTMOS
are cached per WAV in `out/_score_cache.json`, so re-scoring (or adding UTMOS
later) takes seconds instead of re-running ASR. If a 16 GB T4 runs out of memory
here, restart the runtime and run this cell alone: WAVs are already on disk.

In [ ]:
!python phase0/run.py score

## 9. Report + blind A/B + download

`--publish` writes the table between the `RESULTS` markers in
`docs/phase0-hindi-bakeoff.md`. Score the two leading candidates by ear without
knowing which is which; the mapping is in `blind_ratings.csv`.

In [ ]:
!python phase0/run.py report --publish
!python phase0/run.py blind --a cb_mtl_configB --b cb_mtl_default
!cd /content && rm -f phase0_out.zip && zip -qr phase0_out.zip voice_clone/phase0/out voice_clone/docs && ls -lh phase0_out.zip
from google.colab import files
files.download("/content/phase0_out.zip")

## 9. IndicF5 in a second session (optional)

IndicF5 expects `transformers` 4.x while chatterbox pins 5.2.0, so it cannot share
this process. `phase0/out` survives a restart, so:

1. Runtime → Restart session
2. Re-run cell 3 (Drive + harness)
3. Run the cell below (installs transformers 4.46.3 + vocos)
4. Runtime → Restart session again
5. Re-run cell 3, then the render cell, then `score` and `report` once more

`score` re-scores every WAV it finds, so the table then includes IndicF5.

In [ ]:
!pip install -q "transformers==4.46.3" vocos
print("now: Runtime -> Restart session, re-run cell 3, then the render cell below")

In [ ]:
!python phase0/run.py render --reference "$REF" --ref-text "$REF_TEXT" \
  --candidates indicf5
!python phase0/run.py score
!python phase0/run.py report

## 10. GGUF runtime check (Phase 0B)

This proves the Android path, so it runs separately from the PyTorch sweep. It
needs `llama.cpp` + `mybigday/codec.cpp` built with CUDA, plus the Chatterbox
GGUF pair (`chatterbox-mtl-q8_0.gguf` + `codec-q8_0.gguf`).

```bash
git clone https://github.com/ggml-org/llama.cpp
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
cmake --build llama.cpp/build -j
# build mybigday/codec.cpp the same way, then export:
export PHASE0_TTS_CLI=/content/codec.cpp/build/tts-cli
export PHASE0_CB_Q8_BACKBONE=/content/models/chatterbox-mtl-q8_0.gguf
export PHASE0_CB_Q8_CODEC=/content/models/codec-q8_0.gguf
python phase0/run.py render --reference "$REF" --candidates gguf_mtl_q8
python phase0/run.py score
```

## Pitfalls

- **torch mismatch**: the repair cell + one restart is not optional; `doctor`
  must pass before rendering.
- **IndicF5 is gated**: accept the licence on HF, then add a `HF_TOKEN` Colab secret.
- **Do not swap the reference mid-run**: the harness refuses to mix takes
  (SHA-256 in `out/manifest.json`).
- **Colab disconnects**: rendering is resumable, just re-run the same cell.
- **Scoring ASR is faster-whisper, not ainotes**: keeps the run dependency-light.
  `no_timestamps` only matters for the ainotes model.